In [ ]:
# Cell 1 — Install
!pip install -q librosa soundfile numpy scikit-learn

In [ ]:
# Cell 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Imports
import os
import numpy as np
import pandas as pd
import librosa
import pickle
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

print('✅ Imports done!')

In [ ]:
# Cell 4 — Define all paths
BASE      = '/content/drive/MyDrive/Hackenza_MUPlovers/'
AUDIO_DIR = BASE + 'data/processed/'
CHUNK_CSV = BASE + 'data/chunk_index.csv'

# Input dirs
EMB_DIR   = BASE + 'cache/embeddings_1039/'    # WavLM-Large [T, 1024]
PROS_DIR  = BASE + 'cache/features/prosody/'   # prosody [T, 10]
NOISE_DIR = BASE + 'cache/noise/'              # noise [T, 5]

# Output dirs
FEAT_DIR  = BASE + 'cache/features_1039/'              # assembled [T, 1039]
NORM_DIR  = BASE + 'cache/features_normalized_1039/'   # final normalized [T, 1039]

# Create folders
for d in [NOISE_DIR, FEAT_DIR, NORM_DIR]:
    os.makedirs(d, exist_ok=True)

# Load chunk index
chunk_index = pd.read_csv(CHUNK_CSV)

print('EMB_DIR exists   :', os.path.exists(EMB_DIR))
print('PROS_DIR exists  :', os.path.exists(PROS_DIR))
print('AUDIO_DIR exists :', os.path.exists(AUDIO_DIR))
print('Unique files     :', chunk_index['file_id'].nunique())
print('Embeddings count :', len(os.listdir(EMB_DIR)) if os.path.exists(EMB_DIR) else 0)
print('\n✅ Paths defined!')

In [ ]:
# Cell 5 — Verify all embeddings are 1024-dim
print('Verifying embeddings in embeddings_1039/...')

all_emb_files = [f for f in os.listdir(EMB_DIR) if f.endswith('.npy')]
issues        = []

for fname in tqdm(all_emb_files):
    emb = np.load(os.path.join(EMB_DIR, fname))
    if emb.shape[1] != 1024:
        issues.append(f'{fname} → {emb.shape}')

print(f'\nTotal embedding files: {len(all_emb_files)}')
if issues:
    print('❌ Wrong dim files:', issues)
    print('Fix these before continuing!')
else:
    print('✅ All files are [T, 1024] — ready to assemble!')

In [ ]:
# Cell 6 — Define noise extraction
def extract_noise_features(audio_chunk, sr=16000):
    if len(audio_chunk) < 512 or np.max(np.abs(audio_chunk)) < 1e-6:
        return np.zeros(5, dtype=np.float32)

    stft           = np.abs(librosa.stft(audio_chunk, n_fft=512, hop_length=256))
    power_spectrum = stft ** 2

    rms          = librosa.feature.rms(y=audio_chunk, frame_length=512, hop_length=256)[0]
    sorted_rms   = np.sort(rms)
    n            = len(sorted_rms)
    noise_floor  = np.mean(sorted_rms[:max(1, n//10)])
    signal_level = np.mean(sorted_rms[max(1, n*9//10):])
    snr_proxy    = float(np.log1p(signal_level / (noise_floor + 1e-8)))

    power_norm        = power_spectrum / (power_spectrum.sum(axis=0, keepdims=True) + 1e-8)
    entropy_per_frame = -np.sum(power_norm * np.log(power_norm + 1e-8), axis=0)
    spectral_entropy  = float(np.mean(entropy_per_frame))

    flatness          = librosa.feature.spectral_flatness(y=audio_chunk, n_fft=512, hop_length=256)[0]
    spectral_flatness = float(np.mean(flatness))

    centroid          = librosa.feature.spectral_centroid(y=audio_chunk, sr=sr, n_fft=512, hop_length=256)[0]
    spectral_centroid = float(np.mean(centroid))

    bandwidth          = librosa.feature.spectral_bandwidth(y=audio_chunk, sr=sr, n_fft=512, hop_length=256)[0]
    spectral_bandwidth = float(np.mean(bandwidth))

    return np.array([
        snr_proxy, spectral_entropy, spectral_flatness,
        spectral_centroid, spectral_bandwidth
    ], dtype=np.float32)

print('✅ Noise function defined!')

In [ ]:
# Cell 7 — Extract noise for all files
all_file_ids = chunk_index['file_id'].unique()
failed       = []

print(f'Extracting noise for {len(all_file_ids)} files...')

for file_id in tqdm(all_file_ids):
    save_path = os.path.join(NOISE_DIR, f'{file_id}.npy')

    if os.path.exists(save_path):
        continue

    try:
        file_chunks      = chunk_index[chunk_index['file_id'] == file_id]
        chunk_boundaries = list(zip(file_chunks['start_sample'], file_chunks['end_sample']))
        audio, _         = librosa.load(AUDIO_DIR + f'{file_id}.wav', sr=16000, mono=True)
        noise_matrix     = np.stack([
            extract_noise_features(audio[s:e]) for (s, e) in chunk_boundaries
        ], axis=0)
        np.save(save_path, noise_matrix)
    except Exception as e:
        print(f'❌ Failed {file_id}: {e}')
        failed.append(file_id)

print(f'\n✅ Noise done! {len(all_file_ids)-len(failed)}/{len(all_file_ids)} files')
if failed:
    print('❌ Failed:', failed)

In [ ]:
# Cell 8 — Define assembly function
def assemble_features(file_id):
    """
    Concatenates:
    embeddings [T, 1024] + prosody [T, 10] + noise [T, 5]
    → [T, 1039]
    Saves to FEAT_DIR and returns the matrix.
    """
    embeddings = np.load(os.path.join(EMB_DIR,   f'{file_id}.npy'))
    prosody    = np.load(os.path.join(PROS_DIR,   f'{file_id}.npy'))
    noise      = np.load(os.path.join(NOISE_DIR,  f'{file_id}.npy'))

    T_emb  = embeddings.shape[0]
    T_pros = prosody.shape[0]
    T_noi  = noise.shape[0]

    if not (T_emb == T_pros == T_noi):
        raise ValueError(f'T mismatch! emb={T_emb}, pros={T_pros}, noise={T_noi}')

    X = np.concatenate([embeddings, prosody, noise], axis=1)  # [T, 1039]

    assert X.shape[1] == 1039, f'Wrong dim! Got {X.shape[1]}, expected 1039'

    np.save(os.path.join(FEAT_DIR, f'{file_id}.npy'), X)
    return X

print('✅ Assembly function defined!')

In [ ]:
# Cell 9 — Assemble all files → features_1039/
files_to_assemble = [
    f.replace('.npy', '')
    for f in os.listdir(EMB_DIR)
    if f.endswith('.npy')
]

print(f'Assembling {len(files_to_assemble)} files...')
print(f'Output: 1024 + 10 + 5 = 1039 dim')
print(f'Saving to: {FEAT_DIR}\n')

failed = []

for file_id in tqdm(files_to_assemble):
    save_path = os.path.join(FEAT_DIR, f'{file_id}.npy')

    if os.path.exists(save_path):
        continue

    try:
        assemble_features(file_id)
    except Exception as e:
        print(f'❌ {file_id}: {e}')
        failed.append(file_id)

print(f'\n✅ Assembly done!')
print(f'   {len(os.listdir(FEAT_DIR))} files in features_1039/')
if failed:
    print('❌ Failed:', failed)

In [ ]:
# Cell 10 — Normalize and save to features_normalized_1039/
print('Loading all 1039-dim features to fit scaler...')

all_file_ids_norm = [
    f.replace('.npy', '')
    for f in os.listdir(FEAT_DIR)
    if f.endswith('.npy')
]

# Stack all chunks to fit scaler
all_chunks = []
for file_id in tqdm(all_file_ids_norm):
    X = np.load(os.path.join(FEAT_DIR, f'{file_id}.npy'))
    all_chunks.append(X)

all_stacked = np.vstack(all_chunks)
print(f'Total chunks stacked: {all_stacked.shape}')

# Fit scaler
print('Fitting StandardScaler...')
scaler = StandardScaler()
scaler.fit(all_stacked)
print('✅ Scaler fitted!')

# Normalize and save to features_normalized_1039/
print(f'\nNormalizing and saving to features_normalized_1039/...')
for file_id in tqdm(all_file_ids_norm):
    X      = np.load(os.path.join(FEAT_DIR, f'{file_id}.npy'))
    X_norm = scaler.transform(X).astype(np.float32)
    np.save(os.path.join(NORM_DIR, f'{file_id}.npy'), X_norm)

print(f'\n✅ Normalization done!')
print(f'   {len(os.listdir(NORM_DIR))} files in features_normalized_1039/')

In [ ]:
# Cell 11 — Save scaler
scaler_path = BASE + 'cache/scaler_1039.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print('✅ Scaler saved at:', scaler_path)

In [ ]:
# Cell 12 — Final verification
all_norm_files = [f for f in os.listdir(NORM_DIR) if f.endswith('.npy')]
issues         = []
shapes         = []

for fname in tqdm(all_norm_files):
    X = np.load(os.path.join(NORM_DIR, fname))
    if X.shape[1] != 1039:
        issues.append(f'WRONG D: {fname} → {X.shape}')
    else:
        shapes.append(X.shape[0])

sample = np.load(os.path.join(NORM_DIR, all_norm_files[0]))

print(f'\n=== Final Verification ===')
print(f'✅ Total files        : {len(shapes)}/{len(all_norm_files)}')
print(f'✅ Feature dim        : 1039')
print(f'✅ T range            : min={min(shapes)}, max={max(shapes)}')
print(f'✅ Mean (should be ~0): {sample.mean():.4f}')
print(f'✅ Std  (should be ~1): {sample.std():.4f}')

if issues:
    print('\n❌ Issues found:', issues)
else:
    print('\n✅ All good! Ready for H!')
    print('→ Tell H to use:')
    print('   FEATURE_DIR = cache/features_normalized_1039/')
    print('   input_dim   = 1039')